<a href="https://colab.research.google.com/github/nadeemaadi451-droid/NL-to-SQL-Transformer-from-scrach/blob/main/NL_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
!pip install -q datasets
from datasets import load_dataset
ds = load_dataset("b-mc2/sql-create-context", split="train")

ds.shape


README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

(78577, 3)

In [2]:
ds[0]

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56',
 'question': 'How many heads of the departments are older than 56 ?',
 'context': 'CREATE TABLE head (age INTEGER)'}

In [3]:
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, test_ds = split["train"], split["test"]


In [4]:
with open("data.txt", 'w') as f:
  for row in ds:
    f.write(row["context"] + "\n")
    f.write(row["question"] + "\n")
    f.write(row["answer"] + "\n")

In [5]:
SEP = "<sep>"
EOS = "<eos>"

def format_data(train_ds):
  texts = []
  for row in train_ds:
    text = row["context"] + " " + row["question"] + " <sep> " + row["answer"] + " <eos>"
    if len(text) <= 400:
        texts.append(text)
  return texts

texts = format_data(train_ds)

print(len(texts))
print(texts[0])
type(texts)


69549
CREATE TABLE table_name_75 (insurgents VARCHAR, civilians VARCHAR) Name the insurgents for civilians being 49 <sep> SELECT insurgents FROM table_name_75 WHERE civilians = "49" <eos>


list

In [6]:
!pip install -q tokenizers

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
special_tokens = ["<pad>", "<bos>", "<eos>", "<sep>"]
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=5000, special_tokens=special_tokens, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.get_vocab_size()

enc = tokenizer.encode(texts[0])
print(enc.tokens)
print(enc.ids)

dec = tokenizer.decode(enc.ids)
print(dec)

import numpy as np
length = [len(tokenizer.encode(i).ids) for i in texts]
print(len(length))

['CREATE', 'ĠTABLE', 'Ġtable', '_', 'name', '_', '75', 'Ġ(', 'ins', 'ur', 'gent', 's', 'ĠVARCHAR', ',', 'Ġcivil', 'ians', 'ĠVARCHAR', ')', 'ĠName', 'Ġthe', 'Ġin', 's', 'ur', 'gent', 's', 'Ġfor', 'Ġcivil', 'ians', 'Ġbeing', 'Ġ49', 'Ġ', '<sep>', 'ĠSELECT', 'Ġin', 's', 'ur', 'gent', 's', 'ĠFROM', 'Ġtable', '_', 'name', '_', '75', 'ĠWHERE', 'Ġcivil', 'ians', 'Ġ=', 'Ġ"', '49', '"', 'Ġ', '<eos>']
[300, 299, 275, 66, 278, 66, 651, 290, 1273, 466, 4134, 86, 268, 15, 4453, 2554, 268, 12, 545, 291, 386, 86, 466, 4134, 86, 380, 4453, 2554, 1830, 2391, 224, 3, 307, 386, 86, 466, 4134, 86, 304, 275, 66, 278, 66, 651, 312, 4453, 2554, 289, 309, 877, 5, 224, 2]
CREATE TABLE table_name_75 (insurgents VARCHAR, civilians VARCHAR) Name the insurgents for civilians being 49  SELECT insurgents FROM table_name_75 WHERE civilians = "49" 
69549


In [7]:
lengths = [len(tokenizer.encode(t).ids) for t in texts]
print(max(lengths), sum(lengths) / len(lengths))

200 62.388517448130095


In [8]:
context_size = 200
batch_size = 64
num_head = 4
n_embed = 128
n_layer=3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dropout = 0.2

In [9]:
"""Encoder-decoder batching:
- encoder input  (src)    = schema + question          (everything before <sep>)
- decoder input  (dec_in) = <bos> + answer             (shifted right)
- labels         (y)      = answer + <eos>             (shifted left)
Shorter examples are padded; padded labels are -100 so the loss ignores them.
"""

import torch
Pad_id = tokenizer.token_to_id("<pad>")
Bos_id = tokenizer.token_to_id("<bos>")
Eos_id = tokenizer.token_to_id("<eos>")
Sep_id = tokenizer.token_to_id("<sep>")


In [10]:
def get_batch(data, batch_size):
    ix = torch.randint(len(data), (batch_size,))
    seqs = [tokenizer.encode(data[int(i)]).ids for i in ix]

    srcs, tgts = [], []
    for s in seqs:
        sep_pos = s.index(Sep_id)
        srcs.append(s[:sep_pos])                  # schema + question
        tgts.append([Bos_id] + s[sep_pos + 1:])   # <bos> answer <eos>

    src_len = max(len(s) for s in srcs)
    tgt_len = max(len(t) for t in tgts) - 1
     # decoder and encoder input will be different
     # encoder input will be the context+question
     # decoder input will be <bos>+ answer
     # label will be answer+ <eos>
    src = torch.full((batch_size, src_len), Pad_id)
    dec_in = torch.full((batch_size, tgt_len), Pad_id)
    y = torch.full((batch_size, tgt_len), -100)
    #print(src.shape)
    #print( dec_in.shape)
    #print(y.shape)
    for b, (s, t) in enumerate(zip(srcs, tgts)): # b is the batch row
        src[b, :len(s)] = torch.tensor(s)
        dec_in[b, :len(t) - 1] = torch.tensor(t[:-1])
        y[b, :len(t) - 1] = torch.tensor(t[1:])
    return src.to(device), dec_in.to(device), y.to(device)

src, dec_in, y = get_batch(texts, batch_size)
print(src.shape, dec_in.shape, y.shape)
print(src[0])
print(dec_in[0])
print(y[0])

torch.Size([64, 66]) torch.Size([64, 49]) torch.Size([64, 49])
tensor([ 300,  299,  275,   66,  278,   66,  701,  290, 3766,  665,  268,   15,
        1878,  382,   12,  442, 3832,  590,  714,  399, 1023, 1878,   34,  224,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0], device='cuda:0')
tensor([   1,  307, 3832,  304,  275,   66,  278,   66,  701,  312, 1878,  455,
        1023,  224,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0], device='cuda:0')
tensor([ 307, 3832,  304,  275,   66,  278,   66,  701,  312, 1878,  455, 1023,
         224,    2, -100, -100, -100, -100,

In [11]:
import torch.nn as nn
import torch.nn.functional as F
import math

# ---- Decoder self-attention (causal) ----
class Masked_Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))
    self.dropout = nn.Dropout(dropout)
  def forward(self, x):
    B, T, C = x.shape
    wei = self.query(x) @ self.key(x).transpose(-2, -1) / (self.key(x).shape[-1] ** 0.5)  # FIX: query @ key^T
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    out = wei @ self.value(x)
    return out

In [12]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, mask=None):
    B, T, C = x.shape
    wei = self.query(x) @ self.key(x).transpose(-2, -1) / (self.key(x).shape[-1] ** 0.5)
    if mask is not None:                                   # mask: (B, T), 1 = real token
      wei = wei.masked_fill(mask[:, None, :] == 0, float('-inf')) # give infinity to padded values so they don't get any probablity value during softmax
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    out = wei @ self.value(x)
    return out

In [13]:
class MultiHead_Masked(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Masked_Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


In [14]:
class MultiHead(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, mask=None):
    out = torch.cat([h(x, mask) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out

In [15]:
class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
        nn.Dropout(dropout)
    )
  def forward(self, x):
    return self.net(x)

In [16]:
class Cross_Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, enc_out, mask=None):
    q = self.query(x)
    k = self.key(enc_out)
    v = self.value(enc_out)
    wei = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)
    if mask is not None:                    # don't attend to encoder <pad> positions
      wei = wei.masked_fill(mask[:, None, :] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    return wei @ v

In [17]:
class MultiHead_Cross(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Cross_Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, enc_out, mask=None):
    out = torch.cat([h(x, enc_out, mask) for h in self.heads], dim=-1)
    return self.dropout(self.proj(out))


In [18]:
class Encoder(nn.Module):
  def __init__(self, num_head, n_embed):
    super().__init__()
    self.sa = MultiHead(num_head, n_embed // num_head)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)
  def forward(self, x, mask=None):
    x = x + self.sa(self.ln1(x), mask)
    x = x + self.ffwd(self.ln2(x))
    return x

In [19]:
class Decoder(nn.Module):
  def __init__(self, num_head, n_embed):
    super().__init__()
    self.sa = MultiHead_Masked(num_head, n_embed // num_head)
    self.ca = MultiHead_Cross(num_head, n_embed // num_head)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)
    self.ln3 = nn.LayerNorm(n_embed)
  def forward(self, x, enc_out, src_mask=None):
    x = x + self.sa(self.ln1(x))
    x = x + self.ca(self.ln2(x), enc_out, src_mask)
    x = x + self.ffwd(self.ln3(x))
    return x

In [20]:
class Transformer(nn.Module):
    def __init__(self, num_head, n_embed):
        super().__init__()
        self.encoder = nn.ModuleList([Encoder(num_head, n_embed) for _ in range(n_layer)])
        self.decoder = nn.ModuleList([Decoder(num_head, n_embed) for _ in range(n_layer)])
        self.ln_enc = nn.LayerNorm(n_embed)
        self.ln_dec = nn.LayerNorm(n_embed)

    def forward(self, src_emb, tgt_emb, src_mask=None):
        x = src_emb
        for blk in self.encoder:
            x = blk(x, src_mask)
        enc_out = self.ln_enc(x)

        y = tgt_emb
        for blk in self.decoder:
            y = blk(y, enc_out, src_mask)
        return self.ln_dec(y)

In [21]:
class MyLLM(nn.Module):
  def __init__(self, vocab_size, n_embed, context_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embed)
    self.positional_encoding = nn.Embedding(context_size, n_embed)
    self.transformer = Transformer(num_head, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def embed(self, ids):
    pos = torch.arange(ids.shape[1], device=ids.device)
    return self.embedding(ids) + self.positional_encoding(pos)

  def forward(self, src, dec_in, targets=None):
    src_mask = (src != Pad_id)
    x = self.transformer(self.embed(src), self.embed(dec_in), src_mask)
    logits = self.lm_head(x)                      # keep [B, T, C]
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T),
                             ignore_index=-100)   # don't overwrite logits
    return logits, loss

  @torch.no_grad()
  def generate(self, src, max_new_tokens):
    self.eval()
    idx = torch.full((src.shape[0], 1), Bos_id, dtype=torch.long, device=src.device)
    finished = torch.zeros(src.shape[0], dtype=torch.bool, device=src.device)
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -context_size:]
      logits, _ = self(src, idx_cond)
      logits = logits[:, -1, :]
      idx_next = logits.argmax(dim=-1, keepdim=True)   # greedy: repeatable results
      idx = torch.cat((idx, idx_next), dim=1)
      finished |= (idx_next.squeeze(1) == Eos_id)
      if finished.all():
        break
    return idx

model = MyLLM(tokenizer.get_vocab_size(), n_embed, context_size).to(device)
logits, loss = model(src, dec_in, y)
print("Logits shape:", logits.shape)   # [B, tgt_len, vocab_size]
print("Loss:", loss)                   # about 8.5 for a 5000 vocab

Logits shape: torch.Size([64, 49, 5000])
Loss: tensor(8.7202, device='cuda:0', grad_fn=<NllLossBackward0>)


In [22]:
from google.colab import drive
drive.mount('/content/drive')
model = MyLLM(tokenizer.get_vocab_size(), n_embed, context_size).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/sql_model.pt", map_location=device))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<All keys matched successfully>

In [23]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model.to(device)

# Define missing hyperparameters and configurations
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for _ in range(max_iters):
  src, dec_in, y= get_batch(texts, batch_size)
  logits, loss= model(src, dec_in, y)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss)


Using device: cuda
tensor(0.2025, device='cuda:0', grad_fn=<NllLossBackward0>)


In [24]:
torch.save(model.state_dict(), "/content/drive/MyDrive/sql_model.pt")
tokenizer.save("/content/drive/MyDrive/sql_tokenizer.json")
print("saved")

saved


In [25]:
# 1. build the test list (same format as the training list)
test_texts = []
for row in test_ds:
    text = row["context"] + " " + row["question"] + " <sep> " + row["answer"] + " <eos>"
    if len(text) <= 400:
        test_texts.append(text)
print("train examples:", len(texts), "| test examples:", len(test_texts))

# 2. average loss over several random batches
@torch.no_grad()
def eval_loss(data, n_batches=50):
    model.eval()                      # dropout off
    losses = []
    for _ in range(n_batches):
        src, dec_in, y = get_batch(data, batch_size)
        _, loss = model(src, dec_in, y)
        losses.append(loss.item())
    model.train()                     # back to training mode
    return sum(losses) / len(losses)

# 3. compare
train_loss = eval_loss(texts)
test_loss = eval_loss(test_texts)
print(f"train loss: {train_loss:.3f}")
print(f"test loss : {test_loss:.3f}")
print(f"gap       : {test_loss - train_loss:.3f}")

# 4. simple verdict
if test_loss - train_loss > 0.5:
    print("Overfitting: test loss is much higher than train loss")
elif train_loss > 2.0 and test_loss > 2.0:
    print("Underfitting: both losses are still high")
else:
    print("Looks like a good fit")

train examples: 69549 | test examples: 7733
train loss: 0.122
test loss : 0.216
gap       : 0.094
Looks like a good fit


In [26]:
# 1. load the trained model (skip if `model` is already trained in this session)
model = MyLLM(tokenizer.get_vocab_size(), n_embed, context_size).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/sql_model.pt", map_location=device))
model.eval()

# 2. build the test list (same format as training)
test_texts = []
for row in test_ds:
    text = row["context"] + " " + row["question"] + " <sep> " + row["answer"] + " <eos>"
    if len(text) <= 400:
        test_texts.append(text)

# 3. generate SQL for one test example
def predict(text):
    ids = tokenizer.encode(text).ids
    sep_pos = ids.index(Sep_id)
    src = torch.tensor([ids[:sep_pos]], device=device)       # schema + question
    ref_ids = ids[sep_pos + 1:]
    if ref_ids and ref_ids[-1] == Eos_id:
        ref_ids = ref_ids[:-1]                                # drop <eos> from reference

    out = model.generate(src, max_new_tokens=100)[0].tolist()[1:]   # drop <bos>
    if Eos_id in out:
        out = out[:out.index(Eos_id)]                         # cut at first <eos>
    return tokenizer.decode(out), tokenizer.decode(ref_ids)

def normalize(s):
    return " ".join(s.lower().split())

# 4. look at 10 examples
for t in test_texts[:10]:
    pred, ref = predict(t)
    print("PRED:", pred)
    print("REF :", ref)
    print("OK" if normalize(pred) == normalize(ref) else "DIFFERENT")
    print()

# 5. exact-match accuracy on 100 examples
N = 100
correct = 0
fails = []
for t in test_texts[:N]:
    pred, ref = predict(t)
    if normalize(pred) == normalize(ref):
        correct += 1
    else:
        fails.append((pred, ref))
print(f"Exact match: {correct}/{N} = {correct/N:.1%}")

# 6. a few failures to study
for pred, ref in fails[:5]:
    print("PRED:", pred)
    print("REF :", ref)
    print()

PRED:  SELECT venue FROM table_name_50 WHERE away_team = "essendon" 
REF :  SELECT venue FROM table_name_50 WHERE away_team = "essendon" 
OK

PRED:  SELECT MIN(game) FROM table_name_61 WHERE opponent = "phoenix" AND record = "29-17" 
REF :  SELECT MIN(game) FROM table_name_61 WHERE opponent = "phoenix" AND record = "29-17" 
OK

PRED:  SELECT opponent FROM table_name_37 WHERE week = 4 
REF :  SELECT opponent FROM table_name_37 WHERE week = "4" 
DIFFERENT

PRED:  SELECT MAX(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0 
REF :  SELECT SUM(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0 
DIFFERENT

PRED:  SELECT name FROM table_name_83 WHERE nat = "sco" AND transfer_window = "sco" AND moving_to = "greenock morton" 
REF :  SELECT name FROM table_name_83 WHERE nat = "sco" AND transfer_window = "summer" AND moving_to = "greenock morton" 
DIFFERENT

PRED:  SELECT venue FROM table_name_50 WHERE result = "7-1" 
REF :  SELECT venue FROM table_name_50 WH

In [27]:
@torch.no_grad()
def to_sql(context, question, max_new_tokens=150):
    model.eval()
    ids = tokenizer.encode(f"{context} {question} <sep>").ids[:-1]   # drop <sep>, same as training src
    src = torch.tensor([ids], device=device)
    out = model.generate(src, max_new_tokens)[0].tolist()[1:]        # skip <bos>
    if Eos_id in out:
        out = out[:out.index(Eos_id)]
    return tokenizer.decode(out)

context  = "CREATE TABLE employees (id INTEGER, name VARCHAR, salary INTEGER, department VARCHAR)"
question = "What is the average salary of employees in each department?"
print(to_sql(context, question))

 SELECT AVG(T1.id), AVG(T1.salary) FROM employees GROUP BY T1.id 
